# Gene embeddings

This is the entity-level tutorial for genes and genetic perturbations. A gene can be embedded through several views: regulatory DNA sequence, translated protein sequence, text/function descriptions, PPI or dependency tables, and perturbation morphology. The public entry point is always `BioEmbedder.embed(...)`.

The examples keep heavy model calls behind `RUN_REAL_EMBEDDING`. The analysis cells use a small demo AnnData so the notebook remains fast and still shows the plotting and comparison workflow.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, pl, tl

RUN_REAL_EMBEDDING = False
RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

embedder = BioEmbedder(device="auto", organism="human")

genes = ["TP53", "EGFR", "MYC", "BRCA1", "JUN", "STAT3", "IRF1", "CXCL8"]


## 1. Generate gene embeddings through different model families

Use the same API for DNA, protein-derived, text-derived, and morphology-derived gene/action embeddings. Gene features usually attach to `.varm`; perturbation/action embeddings are stored as payloads in `.uns` and can be materialized to `.obsm` when a model needs one vector per observation.


In [ ]:
if RUN_REAL_EMBEDDING:
    dna_payload = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="enformer_human_rough",
        output="payload",
        key="X_gene_dna_enformer",
        region="promoter",
        pooling_strategy="mean",
    )

    protein_payload = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="payload",
        key="X_gene_protein_esm2_650M",
        pooling_strategy="mean",
    )

    text_payload = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="minilm_l6_v2",
        output="payload",
        key="X_gene_text_minilm_l6_v2",
    )

    print(dna_payload["key"], dna_payload["n_entities"], dna_payload["n_dims"])
    print(protein_payload["key"], protein_payload["n_entities"], protein_payload["n_dims"])
    print(text_payload["key"], text_payload["n_entities"], text_payload["n_dims"])
else:
    print("Set RUN_REAL_EMBEDDING=True to run Enformer, ESM-2, and text gene embeddings.")


## 2. Perturbation morphology for gene actions

Morphology is indexed by perturbation, not by measured observations. The canonical source of truth lives in `.uns`; `.obsm` is an explicit projection for downstream models that require one action vector per cell.


In [ ]:
if RUN_REAL_EMBEDDING:
    from embpy.io.exporters import materialize_perturbation_obsm

    pert_adata = ad.AnnData(
        X=np.zeros((len(genes), 1), dtype=np.float32),
        obs=pd.DataFrame({"perturbation": genes}, index=[f"cell_{i}" for i in range(len(genes))]),
        var=pd.DataFrame(index=["placeholder"]),
    )

    pert_adata = embedder.embed(
        genes,
        entity_type="perturbation",
        id_type="symbol",
        model="subcell_mae_rybg",
        output="anndata",
        target=pert_adata,
        attach_to="uns",
        key="X_pert_subcell_mae_rybg",
        morphology_dataset="hpa",
        morphology_source="subcell",
        max_images=3,
        aggregation="mean",
    )

    materialize_perturbation_obsm(
        pert_adata,
        embedding_key="X_pert_subcell_mae_rybg",
        perturbation_key="perturbation",
        obsm_key="X_pert_subcell_mae_rybg",
        missing="nan",
    )

    print(pert_adata.uns["perturbations"].keys())
    print(pert_adata.obsm["X_pert_subcell_mae_rybg"].shape)


## 3. Plot-ready entity AnnData

`embpy.pl` expects row-aligned matrices in `.obsm`, so for entity-level comparisons it is convenient to make a small AnnData with one row per gene. Real payloads from `output="payload"` can be converted in the same way.


In [ ]:
def make_gene_demo(labels: list[str]) -> ad.AnnData:
    groups = pd.Series(
        ["DNA repair", "RTK signaling", "MYC program", "DNA repair", "AP1", "JAK-STAT", "interferon", "cytokine"],
        index=labels,
        name="pathway",
    )
    centers = {name: rng.normal(size=12) for name in groups.unique()}
    X_dna = np.vstack([centers[groups.loc[g]] + rng.normal(scale=0.25, size=12) for g in labels]).astype("float32")
    X_protein = (X_dna @ rng.normal(size=(12, 10)) + rng.normal(scale=0.3, size=(len(labels), 10))).astype("float32")
    X_text = (X_dna @ rng.normal(size=(12, 8)) + rng.normal(scale=0.35, size=(len(labels), 8))).astype("float32")

    obs = pd.DataFrame({"symbol": labels, "pathway": groups.values}, index=labels)
    out = ad.AnnData(X=np.zeros((len(labels), 1), dtype="float32"), obs=obs, var=pd.DataFrame(index=["placeholder"]))
    out.obsm["X_gene_dna"] = X_dna
    out.obsm["X_gene_protein"] = X_protein
    out.obsm["X_gene_text"] = X_text
    return out

gene_space = make_gene_demo(genes)
gene_space


## 4. Annotate and plot the embedding space

Use annotation columns in `.obs` for coloring, faceting, and labeling. In a real run these columns can come from `embpy.tl.annotate_gene_perturbations(...)`, your CRISPR metadata, or any curated table.


In [ ]:
pl.plot_embedding_space(
    gene_space,
    obsm_key="X_gene_dna",
    method="pca",
    color="pathway",
    annotate=True,
    annotate_col="symbol",
    title="Gene regulatory embedding colored by pathway",
)

pl.embedding_color_panel(
    gene_space,
    obsm_key="X_gene_protein",
    method="pca",
    color_keys=["pathway", "symbol"],
    annotate=True,
    annotate_col="symbol",
    title="Protein-derived gene embedding annotations",
)


## 5. Compare model views

Compare neighborhoods, value distributions, and pairwise similarity patterns across embedding spaces.


In [ ]:
_, mean_overlap = tl.compute_knn_overlap(gene_space, "X_gene_dna", "X_gene_protein", k=3)
print(f"Mean DNA/protein KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(gene_space, obsm_keys=["X_gene_dna", "X_gene_protein", "X_gene_text"], k=3)
pl.cross_embedding_correlation(gene_space, "X_gene_dna", "X_gene_protein")
pl.embedding_norms(gene_space, obsm_keys=["X_gene_dna", "X_gene_protein", "X_gene_text"])


## 6. Save reusable outputs

Prefer AnnData/Zarr/CSV outputs for public artifacts. Use payloads when you want the exact `.uns` object for perturbation or action embeddings.


In [ ]:
if RUN_REAL_EMBEDDING:
    embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model=["enformer_human_rough", "esm2_650M", "minilm_l6_v2"],
        output="table",
        fmt="zarr",
        path="gene_embedding_tables.zarr",
        harmonize_dim=64,
    )
